# H15 - Distillation Viability (DistilBERT Student)

This notebook addresses H15.1, H15.2, and H15.3:

- **H15.1:** Use Gemma 4 E2B as teacher to label the H9 corpus with `(scam_prob, suspicious_prob, safe_prob)`, then train a DistilBERT student with KL loss plus `0.1 * CE`.
- **H15.2:** Compare student vs teacher on the held-out test set and check whether student F1 is at least `0.90 * teacher F1`; also check quantized student size is <= 100 MB.
- **H15.3:** Write the decision output for whether DistilBERT is viable as the always-loaded SMS triage tier.

This notebook uses the H9 Google Drive handoff artifacts, so it works across separate Colab runtimes. For the local unblocker flow, H9.5 must have already produced:

```text
/content/drive/MyDrive/GemScan/notebooks/data/processed/h9_local_sms_corpus_scrubbed.csv
```

For a fast smoke test, set `MAX_TEACHER_ROWS` to a small value. For official H15, set `MAX_TEACHER_ROWS = None` and use the official H9 50k-message corpus.


## Install

Run this in a fresh Colab runtime, then restart the runtime before continuing if package imports behave oddly. Use a GPU runtime for the teacher-labeling and student-training steps.


In [ ]:
# Gemma 4 support requires a newer Transformers build than the pinned H8 stack.
# Use the CUDA 12.8 PyTorch index so torch/torchvision/torchaudio resolve to matching builds.
%pip install -q --upgrade --extra-index-url https://download.pytorch.org/whl/cu128 \
  git+https://github.com/huggingface/transformers.git \
  torch \
  torchvision \
  torchaudio \
  accelerate \
  tokenizers \
  huggingface_hub \
  safetensors \
  datasets==2.21.0 \
  scikit-learn==1.5.1 \
  pandas==2.2.2 \
  numpy==1.26.4


## Persistent Paths and Auth

Mount Google Drive so outputs survive separate Colab notebooks. Hugging Face login is required if the Gemma model requires gated access for your account.


In [ ]:
from pathlib import Path

try:
    from google.colab import drive
    drive.mount("/content/drive")
except ModuleNotFoundError:
    pass

NOTEBOOKS_ROOT = Path("/content/drive/MyDrive/GemScan/notebooks")
DATA_DIR = NOTEBOOKS_ROOT / "data"
PROCESSED_DIR = DATA_DIR / "processed"
RESULTS_DIR = NOTEBOOKS_ROOT / "_results"
MODELS_DIR = DATA_DIR / "models"
TEACHER_DIR = DATA_DIR / "teacher_labels"

for directory in [PROCESSED_DIR, RESULTS_DIR, MODELS_DIR, TEACHER_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

CORPUS_PATH = PROCESSED_DIR / "h9_local_sms_corpus_scrubbed.csv"
TEACHER_LABELS_PATH = TEACHER_DIR / "h15_teacher_soft_labels.csv"
STUDENT_OUTPUT_DIR = MODELS_DIR / "distilbert_h15_student"
QUANTIZED_OUTPUT_DIR = MODELS_DIR / "distilbert_h15_student_dynamic_int8"
METRICS_PATH = RESULTS_DIR / "h15_distillation_metrics.json"
DECISION_PATH = RESULTS_DIR / "h15_decision_summary.md"

print("NOTEBOOKS_ROOT", NOTEBOOKS_ROOT)
print("CORPUS_PATH", CORPUS_PATH)
print("TEACHER_LABELS_PATH", TEACHER_LABELS_PATH)


In [ ]:
from huggingface_hub import notebook_login

# Run this if model download fails with an authentication or license error.
# notebook_login()


## Configuration

Use `MAX_TEACHER_ROWS = None` for an official full run. A small integer is only for pipeline debugging.


In [ ]:
import random
import numpy as np
import torch

SEED = 0
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

TEACHER_MODEL_ID = "google/gemma-4-E2B-it"
STUDENT_MODEL_ID = "distilbert-base-uncased"

# For official H15, set this to None and use the official cleaned ~50k H9 corpus.
MAX_TEACHER_ROWS = None

MAX_INPUT_LENGTH = 192
TEACHER_MAX_NEW_TOKENS = 96
TEACHER_TEMPERATURE = 0.0
TEACHER_BATCH_SAVE_EVERY = 25

LABELS = ["safe", "suspicious", "scam"]
label2id = {label: idx for idx, label in enumerate(LABELS)}
id2label = {idx: label for label, idx in label2id.items()}

print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("WARNING: no GPU detected. Full H15 teacher labeling/training will be slow on CPU.")


## Load H9 Scrubbed Corpus

The corpus must contain at least `text` and `label`. Binary SMS corpora map `ham -> safe` and `spam -> scam`; `suspicious` examples are preserved if present.


In [ ]:
import pandas as pd

if not CORPUS_PATH.exists():
    raise FileNotFoundError(
        f"Missing {CORPUS_PATH}. Run h9_local_dataset_seed.ipynb through H9.5 first."
    )

corpus = pd.read_csv(CORPUS_PATH)
print("raw corpus", corpus.shape)
print(corpus.columns.tolist())

label_map = {
    "ham": "safe",
    "legitimate": "safe",
    "safe": "safe",
    "spam": "scam",
    "phishing": "scam",
    "fraud": "scam",
    "scam": "scam",
    "suspicious": "suspicious",
}

corpus["verdict"] = corpus["label"].astype(str).str.lower().str.strip().map(label_map)
corpus = corpus.dropna(subset=["text", "verdict"]).copy()
corpus["text"] = corpus["text"].astype(str).str.strip()
corpus = corpus[corpus["text"].ne("")].copy()
corpus["hard_label"] = corpus["verdict"].map(label2id)

# Keep IDs stable if H9 corpus did not include one.
if "id" not in corpus.columns:
    corpus["id"] = [f"h15-{i:06d}" for i in range(len(corpus))]

corpus = corpus.drop_duplicates(subset=["text", "verdict"]).reset_index(drop=True)

if MAX_TEACHER_ROWS is not None:
    corpus = corpus.sample(n=min(MAX_TEACHER_ROWS, len(corpus)), random_state=SEED).reset_index(drop=True)

print("usable corpus", corpus.shape)
print(corpus["verdict"].value_counts())
if len(corpus) < 50_000:
    print("WARNING: H15.1 calls for a 50k-message corpus. This run is provisional unless the official H9 corpus is used.")

corpus[["id", "source", "language", "text", "verdict", "hard_label"]].head()


## Train / Validation / Test Split

Teacher and student are evaluated on the same held-out split. Teacher soft labels are generated for all splits so the student can train on teacher probabilities and the teacher can be scored on test.


In [ ]:
from sklearn.model_selection import train_test_split

stratify_col = corpus["hard_label"] if corpus["hard_label"].nunique() > 1 else None
train_df, temp_df = train_test_split(
    corpus,
    test_size=0.30,
    random_state=SEED,
    stratify=stratify_col,
)

temp_stratify = temp_df["hard_label"] if temp_df["hard_label"].nunique() > 1 else None
val_df, test_df = train_test_split(
    temp_df,
    test_size=0.50,
    random_state=SEED,
    stratify=temp_stratify,
)

for name, frame in [("train", train_df), ("val", val_df), ("test", test_df)]:
    frame.loc[:, "split"] = name

split_df = pd.concat([train_df, val_df, test_df], ignore_index=True)
print(split_df.groupby(["split", "verdict"]).size())


## H15.1 - Gemma E2B Teacher Soft Labels

This step asks Gemma E2B to return calibrated-ish class probabilities. The notebook saves after every `TEACHER_BATCH_SAVE_EVERY` rows so interrupted runs can resume.

Expected output columns:

```text
safe_prob, suspicious_prob, scam_prob, teacher_verdict, teacher_raw, teacher_parse_ok
```


In [ ]:
import json
import math
import re
from typing import Dict

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

teacher_tokenizer = AutoTokenizer.from_pretrained(TEACHER_MODEL_ID)
teacher_model = AutoModelForCausalLM.from_pretrained(
    TEACHER_MODEL_ID,
    torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
    device_map="auto" if torch.cuda.is_available() else None,
    low_cpu_mem_usage=True,
)
teacher_model.eval()

print("teacher loaded", TEACHER_MODEL_ID)


In [ ]:
def teacher_prompt(text: str) -> str:
    return f"""Classify this SMS/message for scam risk.
Return exactly one JSON object and nothing else.
The probabilities must sum to 1.0.

Allowed verdicts: safe, suspicious, scam.

Message:
<<<
{text}
>>>

Required JSON:
{{"safe_prob": 0.0, "suspicious_prob": 0.0, "scam_prob": 0.0, "verdict": "safe"}}
"""


def extract_json_object(raw: str) -> Dict:
    match = re.search(r"\{.*\}", raw, flags=re.DOTALL)
    if not match:
        raise ValueError("No JSON object found")
    return json.loads(match.group(0))


def normalize_probs(parsed: Dict) -> Dict:
    probs = {
        "safe_prob": float(parsed.get("safe_prob", 0.0)),
        "suspicious_prob": float(parsed.get("suspicious_prob", 0.0)),
        "scam_prob": float(parsed.get("scam_prob", 0.0)),
    }
    probs = {key: max(0.0, min(1.0, value)) for key, value in probs.items() if math.isfinite(value)}
    total = sum(probs.values())
    if total <= 0:
        raise ValueError("Teacher probabilities sum to zero")
    probs = {key: value / total for key, value in probs.items()}
    verdict = str(parsed.get("verdict", "")).lower().strip()
    if verdict not in LABELS:
        verdict = max(
            [("safe", probs["safe_prob"]), ("suspicious", probs["suspicious_prob"]), ("scam", probs["scam_prob"])],
            key=lambda item: item[1],
        )[0]
    return {**probs, "teacher_verdict": verdict}


def fallback_probs_from_text(raw: str) -> Dict:
    lowered = raw.lower()
    verdict = None
    if "scam" in lowered:
        verdict = "scam"
    elif "suspicious" in lowered:
        verdict = "suspicious"
    elif "safe" in lowered:
        verdict = "safe"

    if verdict is None:
        raise ValueError("No fallback verdict keyword found")

    # Smoothed one-hot fallback. Marked as fallback, not strict JSON parse.
    fallback = {
        "safe": {"safe_prob": 0.90, "suspicious_prob": 0.08, "scam_prob": 0.02},
        "suspicious": {"safe_prob": 0.10, "suspicious_prob": 0.80, "scam_prob": 0.10},
        "scam": {"safe_prob": 0.02, "suspicious_prob": 0.08, "scam_prob": 0.90},
    }[verdict]
    return {**fallback, "teacher_verdict": verdict}


def parse_teacher_output(raw: str) -> Dict:
    try:
        parsed = extract_json_object(raw)
        normalized = normalize_probs(parsed)
        return {**normalized, "teacher_parse_ok": True, "teacher_usable": True, "teacher_parse_method": "json"}
    except Exception as json_exc:
        try:
            fallback = fallback_probs_from_text(raw)
            return {**fallback, "teacher_parse_ok": False, "teacher_usable": True, "teacher_parse_method": "keyword_fallback", "teacher_parse_error": repr(json_exc)}
        except Exception as fallback_exc:
            return {
                "safe_prob": np.nan,
                "suspicious_prob": np.nan,
                "scam_prob": np.nan,
                "teacher_verdict": "parse_failed",
                "teacher_parse_ok": False,
                "teacher_usable": False,
                "teacher_parse_method": "failed",
                "teacher_parse_error": f"json={repr(json_exc)}; fallback={repr(fallback_exc)}",
            }


def encode_teacher_prompt(prompt: str):
    messages = [{"role": "user", "content": prompt}]
    try:
        return teacher_tokenizer.apply_chat_template(
            messages,
            add_generation_prompt=True,
            return_tensors="pt",
            return_dict=True,
        )
    except Exception:
        return teacher_tokenizer(prompt, return_tensors="pt", truncation=True, max_length=1024)


def teacher_label_one(text: str) -> Dict:
    prompt = teacher_prompt(text)
    inputs = encode_teacher_prompt(prompt)
    inputs = {key: value.to(teacher_model.device) for key, value in inputs.items()}

    with torch.inference_mode():
        output = teacher_model.generate(
            **inputs,
            max_new_tokens=TEACHER_MAX_NEW_TOKENS,
            do_sample=False,
            pad_token_id=teacher_tokenizer.eos_token_id,
        )

    generated = output[0][inputs["input_ids"].shape[-1]:]
    raw = teacher_tokenizer.decode(generated, skip_special_tokens=True).strip()
    parsed = parse_teacher_output(raw)
    return {**parsed, "teacher_raw": raw}


In [ ]:
# Resume support: keep completed teacher rows and label only missing IDs.
# If you previously produced rows with zero usable labels, delete TEACHER_LABELS_PATH or set RESET_TEACHER_LABELS=True.
RESET_TEACHER_LABELS = False

if RESET_TEACHER_LABELS and TEACHER_LABELS_PATH.exists():
    TEACHER_LABELS_PATH.unlink()
    print("deleted existing teacher labels", TEACHER_LABELS_PATH)

if TEACHER_LABELS_PATH.exists():
    teacher_df = pd.read_csv(TEACHER_LABELS_PATH)
    completed_ids = set(teacher_df["id"].astype(str))
    teacher_rows = teacher_df.to_dict("records")
    print("resuming existing teacher labels", teacher_df.shape)
else:
    completed_ids = set()
    teacher_rows = []

rows_to_label = split_df[~split_df["id"].astype(str).isin(completed_ids)].reset_index(drop=True)
print("rows remaining for teacher labeling", len(rows_to_label))

for idx, row in rows_to_label.iterrows():
    try:
        label_result = teacher_label_one(row["text"])
    except Exception as exc:
        label_result = {
            "safe_prob": np.nan,
            "suspicious_prob": np.nan,
            "scam_prob": np.nan,
            "teacher_verdict": "generation_failed",
            "teacher_raw": "",
            "teacher_parse_ok": False,
            "teacher_usable": False,
            "teacher_parse_method": "generation_failed",
            "teacher_parse_error": repr(exc),
        }

    teacher_rows.append(
        {
            "id": row["id"],
            "split": row["split"],
            "source": row.get("source", "unknown"),
            "language": row.get("language", "unknown"),
            "text": row["text"],
            "hard_verdict": row["verdict"],
            "hard_label": row["hard_label"],
            **label_result,
        }
    )

    if (idx + 1) % TEACHER_BATCH_SAVE_EVERY == 0:
        pd.DataFrame(teacher_rows).to_csv(TEACHER_LABELS_PATH, index=False)
        print("saved", len(teacher_rows), TEACHER_LABELS_PATH)

teacher_df = pd.DataFrame(teacher_rows)
teacher_df.to_csv(TEACHER_LABELS_PATH, index=False)
print("teacher labels saved", teacher_df.shape, TEACHER_LABELS_PATH)
print(teacher_df["teacher_usable"].value_counts(dropna=False) if "teacher_usable" in teacher_df else "teacher_usable column missing")
print(teacher_df["teacher_parse_method"].value_counts(dropna=False) if "teacher_parse_method" in teacher_df else "teacher_parse_method column missing")
teacher_df.head()


## Prepare Distillation Dataset

Rows with teacher parse failures are excluded from student training. If too many rows fail, fix the teacher prompt/parser before continuing.


In [ ]:
teacher_df = pd.read_csv(TEACHER_LABELS_PATH)

if "teacher_usable" not in teacher_df.columns:
    teacher_df["teacher_usable"] = teacher_df.get("teacher_parse_ok", False)

print("Teacher label parse diagnostics")
print(teacher_df["teacher_usable"].value_counts(dropna=False))
if "teacher_parse_method" in teacher_df.columns:
    print(teacher_df["teacher_parse_method"].value_counts(dropna=False))

if not teacher_df[teacher_df["teacher_usable"] == True].empty:
    display(teacher_df[teacher_df["teacher_usable"] == True][["teacher_verdict", "safe_prob", "suspicious_prob", "scam_prob", "teacher_raw"]].head(5))
else:
    print("No usable labels yet. Example failures:")
    cols = [c for c in ["teacher_verdict", "teacher_parse_error", "teacher_raw"] if c in teacher_df.columns]
    display(teacher_df[cols].head(10))

valid_teacher_df = teacher_df[teacher_df["teacher_usable"] == True].copy()

for column in ["safe_prob", "suspicious_prob", "scam_prob"]:
    valid_teacher_df[column] = pd.to_numeric(valid_teacher_df[column], errors="coerce")

valid_teacher_df = valid_teacher_df.dropna(subset=["safe_prob", "suspicious_prob", "scam_prob", "hard_label", "text"])
valid_teacher_df["hard_label"] = valid_teacher_df["hard_label"].astype(int)
valid_teacher_df["teacher_label"] = valid_teacher_df[["safe_prob", "suspicious_prob", "scam_prob"]].values.argmax(axis=1)

print("teacher rows", teacher_df.shape)
print("valid teacher rows", valid_teacher_df.shape)
print(valid_teacher_df.groupby(["split", "hard_verdict"]).size())

if valid_teacher_df.empty:
    raise RuntimeError("No usable teacher labels. Set RESET_TEACHER_LABELS=True and rerun teacher labeling after the parser/prompt fix.")


In [ ]:
from datasets import Dataset
from transformers import AutoTokenizer

student_tokenizer = AutoTokenizer.from_pretrained(STUDENT_MODEL_ID)

student_columns = ["text", "hard_label", "safe_prob", "suspicious_prob", "scam_prob"]
train_student_df = valid_teacher_df[valid_teacher_df["split"] == "train"][student_columns].copy()
val_student_df = valid_teacher_df[valid_teacher_df["split"] == "val"][student_columns].copy()
test_student_df = valid_teacher_df[valid_teacher_df["split"] == "test"].copy()

train_ds = Dataset.from_pandas(train_student_df, preserve_index=False)
val_ds = Dataset.from_pandas(val_student_df, preserve_index=False)

def tokenize_for_student(batch):
    tokenized = student_tokenizer(
        batch["text"],
        truncation=True,
        padding="max_length",
        max_length=MAX_INPUT_LENGTH,
    )
    tokenized["labels"] = batch["hard_label"]
    tokenized["teacher_probs"] = [
        [safe, suspicious, scam]
        for safe, suspicious, scam in zip(batch["safe_prob"], batch["suspicious_prob"], batch["scam_prob"])
    ]
    return tokenized

train_ds = train_ds.map(tokenize_for_student, batched=True, remove_columns=train_ds.column_names)
val_ds = val_ds.map(tokenize_for_student, batched=True, remove_columns=val_ds.column_names)
train_ds.set_format("torch")
val_ds.set_format("torch")

print(train_ds)
print(val_ds)


## H15.1 - Train DistilBERT With KL + 0.1 CE

This is the actual distillation loss required by H15.1. `teacher_probs` provide the soft-label target for KL divergence, and `labels` provide the hard-label target for the `0.1 * CE` term.


In [ ]:
import torch.nn.functional as F
from transformers import AutoModelForSequenceClassification, Trainer, TrainingArguments

student_model = AutoModelForSequenceClassification.from_pretrained(
    STUDENT_MODEL_ID,
    num_labels=len(LABELS),
    id2label=id2label,
    label2id=label2id,
)

class DistillationTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        teacher_probs = inputs.pop("teacher_probs").to(model.device).float()
        labels = inputs.get("labels").to(model.device)
        outputs = model(**inputs)
        logits = outputs.logits
        log_probs = F.log_softmax(logits, dim=-1)
        kl_loss = F.kl_div(log_probs, teacher_probs, reduction="batchmean")
        ce_loss = F.cross_entropy(logits, labels)
        loss = kl_loss + 0.1 * ce_loss
        return (loss, outputs) if return_outputs else loss

args = TrainingArguments(
    output_dir=str(STUDENT_OUTPUT_DIR),
    evaluation_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    num_train_epochs=3,
    weight_decay=0.01,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    report_to="none",
    seed=SEED,
)

trainer = DistillationTrainer(
    model=student_model,
    args=args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    tokenizer=student_tokenizer,
)

trainer.train()


## H15.2 - Compare Student vs Teacher

The target is `student_macro_f1 >= 0.90 * teacher_macro_f1` on the held-out test set.


In [ ]:
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score, classification_report, confusion_matrix

student_test_ds = Dataset.from_pandas(
    test_student_df[student_columns].copy(),
    preserve_index=False,
)
student_test_ds = student_test_ds.map(tokenize_for_student, batched=True, remove_columns=student_test_ds.column_names)
student_test_ds.set_format("torch")

pred_output = trainer.predict(student_test_ds)
student_preds = pred_output.predictions.argmax(axis=1)
hard_labels = test_student_df["hard_label"].astype(int).to_numpy()
teacher_preds = test_student_df["teacher_label"].astype(int).to_numpy()

def metric_block(prefix, preds, labels):
    return {
        f"{prefix}_accuracy": accuracy_score(labels, preds),
        f"{prefix}_macro_f1": f1_score(labels, preds, average="macro", zero_division=0),
        f"{prefix}_scam_f1": f1_score(labels, preds, labels=[label2id["scam"]], average="macro", zero_division=0),
        f"{prefix}_scam_precision": precision_score(labels, preds, labels=[label2id["scam"]], average="macro", zero_division=0),
        f"{prefix}_scam_recall": recall_score(labels, preds, labels=[label2id["scam"]], average="macro", zero_division=0),
    }

metrics = {}
metrics.update(metric_block("teacher", teacher_preds, hard_labels))
metrics.update(metric_block("student", student_preds, hard_labels))
metrics["student_to_teacher_macro_f1_ratio"] = (
    metrics["student_macro_f1"] / metrics["teacher_macro_f1"]
    if metrics["teacher_macro_f1"] > 0 else 0.0
)
metrics["student_passes_90pct_teacher_f1"] = metrics["student_to_teacher_macro_f1_ratio"] >= 0.90
metrics["test_rows"] = int(len(test_student_df))
metrics["train_rows"] = int(len(train_student_df))
metrics["val_rows"] = int(len(val_student_df))
metrics["teacher_rows_total"] = int(len(teacher_df))
metrics["teacher_rows_valid"] = int(len(valid_teacher_df))

print(json.dumps(metrics, indent=2))
print()
print("Teacher report vs hard labels")
print(classification_report(hard_labels, teacher_preds, target_names=LABELS, zero_division=0))
print()
print("Student report vs hard labels")
print(classification_report(hard_labels, student_preds, target_names=LABELS, zero_division=0))
print()
print("Student confusion matrix")
print(confusion_matrix(hard_labels, student_preds, labels=[0, 1, 2]))


## H15.2 - Save Student and Check Size

The production requirement is a student size <= 100 MB after quantization. This cell saves the normal Trainer model and a dynamic int8 state dict for size checking.


In [ ]:
import os
import json
import torch

STUDENT_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
trainer.save_model(str(STUDENT_OUTPUT_DIR))
student_tokenizer.save_pretrained(str(STUDENT_OUTPUT_DIR))

def directory_size_bytes(path: Path) -> int:
    return sum(file.stat().st_size for file in path.rglob("*") if file.is_file())

fp32_size_bytes = directory_size_bytes(STUDENT_OUTPUT_DIR)

quantized_model = torch.quantization.quantize_dynamic(
    trainer.model.cpu(),
    {torch.nn.Linear},
    dtype=torch.qint8,
)
QUANTIZED_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
quantized_state_path = QUANTIZED_OUTPUT_DIR / "pytorch_model_dynamic_int8_state_dict.pt"
torch.save(quantized_model.state_dict(), quantized_state_path)
student_tokenizer.save_pretrained(str(QUANTIZED_OUTPUT_DIR))

quantized_size_bytes = directory_size_bytes(QUANTIZED_OUTPUT_DIR)
metrics["student_fp32_size_bytes"] = int(fp32_size_bytes)
metrics["student_dynamic_int8_size_bytes"] = int(quantized_size_bytes)
metrics["student_dynamic_int8_size_mb"] = round(quantized_size_bytes / 1_000_000, 2)
metrics["student_size_passes_100mb"] = quantized_size_bytes <= 100_000_000

print("fp32 model MB", round(fp32_size_bytes / 1_000_000, 2))
print("dynamic int8 model MB", metrics["student_dynamic_int8_size_mb"])
print("size passes <=100MB", metrics["student_size_passes_100mb"])


## H15.3 - Decision Output

This writes the decision summary expected by H15.3. If either F1 ratio or size target fails, DistilBERT should not be promoted to the always-loaded SMS triage tier without revising the spec or rerunning with better data/training.


In [ ]:
import json
from datetime import datetime, timezone

metrics["h15_complete_for_official_claim"] = bool(
    metrics["student_passes_90pct_teacher_f1"]
    and metrics["student_size_passes_100mb"]
    and len(corpus) >= 50_000
)
metrics["timestamp_utc"] = datetime.now(timezone.utc).isoformat()
metrics["teacher_model_id"] = TEACHER_MODEL_ID
metrics["student_model_id"] = STUDENT_MODEL_ID
metrics["corpus_path"] = str(CORPUS_PATH)
metrics["teacher_labels_path"] = str(TEACHER_LABELS_PATH)

with open(METRICS_PATH, "w") as f:
    json.dump(metrics, f, indent=2)

if metrics["student_passes_90pct_teacher_f1"] and metrics["student_size_passes_100mb"]:
    decision = "PASS: DistilBERT student is viable for the always-loaded SMS triage tier, subject to official corpus size/review."
else:
    decision = "FAIL: DistilBERT student does not yet clear H15 viability gates. Revise data, teacher labels, training, or spec."

if len(corpus) < 50_000:
    decision += " This run used fewer than 50k rows, so it is provisional and cannot complete official H15."

summary = f"""# H15 Distillation Decision

- Timestamp UTC: {metrics['timestamp_utc']}
- Teacher model: `{TEACHER_MODEL_ID}`
- Student model: `{STUDENT_MODEL_ID}`
- Corpus rows: {len(corpus)}
- Valid teacher rows: {metrics['teacher_rows_valid']} / {metrics['teacher_rows_total']}
- Teacher macro-F1: {metrics['teacher_macro_f1']:.4f}
- Student macro-F1: {metrics['student_macro_f1']:.4f}
- Student/teacher macro-F1 ratio: {metrics['student_to_teacher_macro_f1_ratio']:.4f}
- F1 gate passed: {metrics['student_passes_90pct_teacher_f1']}
- Dynamic int8 student size: {metrics['student_dynamic_int8_size_mb']} MB
- Size gate passed: {metrics['student_size_passes_100mb']}

## Decision

{decision}
"""

DECISION_PATH.write_text(summary)
print(summary)
print("saved metrics", METRICS_PATH)
print("saved decision", DECISION_PATH)
